# Adaptación del dataset original (Favorita) a los esquemas reales del SPC

**Objetivo.** Probar el sistema con datos de base real. El dataset original `data_smote/` es el de
Kaggle **Corporación Favorita Grocery Sales Forecasting** (retail Ecuador). Aquí lo transformamos a
los **tres esquemas de dominio** del SPC (`ventas`, `compras`, `almacen`), definidos en
`src/spc/synthetic/esquemas.py` (fuente única de la verdad), y generamos archivos de prueba
compatibles con la API (`/v2`, `/v3`), Excel y JSON.

**Reglas que respeta este notebook**
- El problema real es **regresión / serie temporal** (target continuo). **SMOTE NO se aplica al
  target continuo** ni a fechas/identificadores; solo a la **etiqueta binaria derivada**
  (`demanda_alta`), **tras el corte temporal y solo en TRAIN**.
- **Anti-fuga:** todo umbral/estadístico (P75, medias, imputaciones, SMOTE) se ajusta **solo en TRAIN**.
- Los campos sin fuente en Favorita se rellenan con **proxies claramente marcados** (REAL / DERIVADO /
  SINTÉTICO) en el `MANIFIESTO.csv`. Nunca se presentan como observaciones reales.

**Marca de procedencia por campo**
- **REAL**: viene directo de Favorita.
- **DERIVADO**: calculado a partir de señal real (calendario, agregación de demanda, política documentada).
- **SINTÉTICO**: sin fuente en Favorita; valor neutro/determinista (semilla 42), sin señal predictiva.


In [1]:
# === 1. Configuración ===
from __future__ import annotations
import sys, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

# Raíz del repo (este notebook vive en notebooks/).
ROOT = Path.cwd()
if not (ROOT / "src" / "spc").exists() and (ROOT.parent / "src" / "spc").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from spc.synthetic.esquemas import esquema_de, validar_conforme, VENTAS, COMPRAS, ALMACEN

SEED = 42
rng_global = np.random.default_rng(SEED)

DATA_SMOTE = ROOT / "data_smote"
OUT_DATOS = ROOT / "data" / "favorita_real"
OUT_EJEMPLOS = ROOT / "examples" / "favorita_real"
OUT_DATOS.mkdir(parents=True, exist_ok=True)
OUT_EJEMPLOS.mkdir(parents=True, exist_ok=True)

# ---- Parámetros de submuestra (documentados) ----
# Ventana temporal reciente (conserva estacionalidad semanal/anual y el desbalance de demanda_alta).
FECHA_INI = "2017-01-01"
FECHA_FIN = "2017-08-15"       # última fecha de train.csv
# Nº de tiendas por tipo (A-E) para lograr una mezcla representativa.
TIENDAS_POR_TIPO = 2
# Cubo temporal para COMPRAS (reposición): semanal.
CUBO_COMPRAS = "W-MON"

print("ROOT:", ROOT)
print("data_smote existe:", DATA_SMOTE.exists())
print("Ventana:", FECHA_INI, "->", FECHA_FIN, "| tiendas/tipo:", TIENDAS_POR_TIPO)


ROOT: D:\UPAO\IX\Taller Integrador I\sistema_prediccion_comercializacion
data_smote existe: True
Ventana: 2017-01-01 -> 2017-08-15 | tiendas/tipo: 2


In [2]:
# === 2. Carga eficiente ===
# 2a. CSV pequeños completos.
items = pd.read_csv(DATA_SMOTE / "items" / "items.csv")             # item_nbr, family, class, perishable
stores = pd.read_csv(DATA_SMOTE / "stores" / "stores.csv")          # store_nbr, city, state, type, cluster
oil = pd.read_csv(DATA_SMOTE / "oil" / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(DATA_SMOTE / "holidays_events" / "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(DATA_SMOTE / "transactions" / "transactions.csv", parse_dates=["date"])
print("items", items.shape, "| stores", stores.shape, "| oil", oil.shape,
      "| holidays", holidays.shape, "| transactions", transactions.shape)

# 2b. Selección de tiendas: TIENDAS_POR_TIPO por cada type (A-E), determinista.
tiendas_sel = (stores.sort_values(["type", "store_nbr"])
                     .groupby("type", group_keys=False)
                     .head(TIENDAS_POR_TIPO))
STORES = sorted(tiendas_sel["store_nbr"].tolist())
print("Tiendas seleccionadas:", STORES)
print(tiendas_sel[["store_nbr", "city", "state", "type", "cluster"]].to_string(index=False))


items (4100, 4) | stores (54, 5) | oil (1218, 2) | holidays (350, 6) | transactions (83488, 3)
Tiendas seleccionadas: [1, 2, 9, 10, 11, 12, 28, 29, 44, 45]
 store_nbr      city     state type  cluster
        44     Quito Pichincha    A        5
        45     Quito Pichincha    A       11
         9     Quito Pichincha    B        6
        11   Cayambe Pichincha    B        6
        10     Quito Pichincha    C       15
        12 Latacunga  Cotopaxi    C       15
         1     Quito Pichincha    D       13
         2     Quito Pichincha    D       13
        28 Guayaquil    Guayas    E       10
        29 Guayaquil    Guayas    E       10


In [3]:
# 2c. Lectura de train.csv (~5 GB) por chunks, filtrando por tienda + ventana.
# train.csv está ordenado por fecha ascendente; cortamos la lectura al superar FECHA_FIN.
# onpromotion se fuerza a 'object' (string): en los años tempranos viene en blanco y pandas
# lo infiere float en unos chunks y object en otros -> el concat multichunk de pandas 3.0 falla.
# Con dtype fijo todos los chunks coinciden y se imputa después (celda 3).
dtypes_train = {"store_nbr": "int16", "item_nbr": "int32", "unit_sales": "float32", "onpromotion": "object"}
usecols = ["date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"]
store_set = set(STORES)
partes = []
leidas = 0
for chunk in pd.read_csv(DATA_SMOTE / "train" / "train.csv", usecols=usecols,
                         dtype=dtypes_train, chunksize=5_000_000):
    leidas += len(chunk)
    # Filtro barato por string ISO antes de parsear fechas.
    m = (chunk["date"] >= FECHA_INI) & (chunk["date"] <= FECHA_FIN) & chunk["store_nbr"].isin(store_set)
    if m.any():
        partes.append(chunk.loc[m].copy())
    # Corte temprano: si el chunk ya pasó la ventana, no hay más datos útiles.
    if chunk["date"].iloc[-1] > FECHA_FIN:
        break

train = pd.concat(partes, ignore_index=True)
del partes
train["date"] = pd.to_datetime(train["date"])
print(f"Filas escaneadas: {leidas:,} | filas retenidas: {len(train):,}")
print(train.head(3).to_string(index=False))


Filas escaneadas: 125,497,040 | filas retenidas: 4,713,306
      date  store_nbr  item_nbr  unit_sales onpromotion
2017-01-02          1    103520         1.0       False
2017-01-02          1    105575         3.0       False
2017-01-02          1    105577         1.0       False


## 3. Validación de calidad (sobre la submuestra)

In [4]:
# === 3. Calidad ===
rep = {}
rep["filas"] = len(train)
rep["rango_fechas"] = (train["date"].min(), train["date"].max())
rep["neg_unit_sales"] = int((train["unit_sales"] < 0).sum())
rep["cero_unit_sales"] = int((train["unit_sales"] == 0).sum())
rep["onpromotion_nulos"] = int(train["onpromotion"].isna().sum())
rep["dup_date_store_item"] = int(train.duplicated(["date", "store_nbr", "item_nbr"]).sum())
rep["items_sin_catalogo"] = int((~train["item_nbr"].isin(items["item_nbr"])).sum())
rep["oil_nulos"] = int(oil["dcoilwtico"].isna().sum())
for k, v in rep.items():
    print(f"{k:24s}: {v}")

# onpromotion: bool/NaN -> {0,1} + bandera de faltante (imputación documentada).
train["onpromotion_missing"] = train["onpromotion"].isna().astype("int8")
train["onpromotion"] = (train["onpromotion"].map({True: 1, False: 0, "True": 1, "False": 0})
                        .fillna(0).astype("int8"))
print("\nonpromotion imputado -> valores:", sorted(train['onpromotion'].unique().tolist()),
      "| faltantes marcados:", int(train['onpromotion_missing'].sum()))


filas                   : 4713306
rango_fechas            : (Timestamp('2017-01-02 00:00:00'), Timestamp('2017-08-15 00:00:00'))
neg_unit_sales          : 311
cero_unit_sales         : 0
onpromotion_nulos       : 0
dup_date_store_item     : 0
items_sin_catalogo      : 0
oil_nulos               : 43



onpromotion imputado -> valores: [0, 1] | faltantes marcados: 0


In [5]:
# === 4. Registro de la submuestra ===
resumen_submuestra = {
    "fecha_ini": FECHA_INI, "fecha_fin": FECHA_FIN,
    "tiendas": STORES, "tiendas_por_tipo": TIENDAS_POR_TIPO,
    "n_filas_train": int(len(train)),
    "n_skus": int(train["item_nbr"].nunique()),
    "criterio": "TIENDAS_POR_TIPO tiendas por cada type (A-E) + ventana temporal reciente; semilla 42",
}
print(json.dumps(resumen_submuestra, indent=2, ensure_ascii=False, default=str))


{
  "fecha_ini": "2017-01-01",
  "fecha_fin": "2017-08-15",
  "tiendas": [
    1,
    2,
    9,
    10,
    11,
    12,
    28,
    29,
    44,
    45
  ],
  "tiendas_por_tipo": 2,
  "n_filas_train": 4713306,
  "n_skus": 3995,
  "criterio": "TIENDAS_POR_TIPO tiendas por cada type (A-E) + ventana temporal reciente; semilla 42"
}


In [6]:
# === 5. Integración de fuentes + calendario ===
# 5a. Catálogo de productos (familia) y tiendas (city/state/type).
df = train.merge(items[["item_nbr", "family", "class"]], on="item_nbr", how="left")
df = df.merge(stores[["store_nbr", "city", "state", "type"]], on="store_nbr", how="left")

# 5b. Petróleo reindexado a calendario diario + ffill/bfill (imputación documentada).
cal = pd.DataFrame({"date": pd.date_range(df["date"].min(), df["date"].max(), freq="D")})
oil_full = cal.merge(oil, on="date", how="left")
oil_full["dcoilwtico"] = oil_full["dcoilwtico"].ffill().bfill()
df = df.merge(oil_full, on="date", how="left")

# 5c. Feriados EFECTIVOS nacionales (excluye transferred=True y 'Work Day'; incluye 'Transfer' como el día real).
h = holidays.copy()
es_efectivo = (h["type"].isin(["Holiday", "Additional", "Bridge", "Event", "Transfer"])
               & ~((h["type"] == "Holiday") & (h["transferred"] == True)))
feriados_nac = pd.to_datetime(
    sorted(h.loc[es_efectivo & (h["locale"] == "National"), "date"].dt.normalize().unique())
)

def dias_a_proximo_feriado(fechas: pd.Series, feriados: pd.DatetimeIndex, cap: int = 30) -> np.ndarray:
    f = pd.to_datetime(fechas).values.astype("datetime64[D]")
    hs = np.array(feriados.values, dtype="datetime64[D]")
    idx = np.searchsorted(hs, f, side="left")   # primer feriado >= fecha
    idx = np.clip(idx, 0, len(hs) - 1)
    d = (hs[idx] - f).astype("timedelta64[D]").astype("int64")
    d[d < 0] = cap                              # sin feriado por delante
    return np.clip(d, 0, cap).astype("int64")

# 5d. Calendario.
df["es_fin_de_semana"] = df["date"].dt.dayofweek.isin([5, 6]).astype("int8")
df["dias_a_proximo_feriado"] = dias_a_proximo_feriado(df["date"], feriados_nac)
df["_month"] = df["date"].dt.month.astype("int8")
df["_dow"] = df["date"].dt.dayofweek.astype("int8")
print("Integrado:", df.shape, "| feriados nacionales efectivos:", len(feriados_nac))
print(df[["date", "store_nbr", "item_nbr", "family", "es_fin_de_semana", "dias_a_proximo_feriado"]].head(3).to_string(index=False))


Integrado: (4713306, 16) | feriados nacionales efectivos: 155
      date  store_nbr  item_nbr    family  es_fin_de_semana  dias_a_proximo_feriado
2017-01-02          1    103520 GROCERY I                 0                       0
2017-01-02          1    105575 GROCERY I                 0                       0
2017-01-02          1    105577 GROCERY I                 0                       0


In [7]:
# === Helpers deterministas para columnas SINTÉTICAS (semilla 42, estables por clave) ===
def _hash_u32(*parts) -> int:
    s = "|".join(str(p) for p in parts)
    return int(hashlib.blake2b(s.encode(), digest_size=4).hexdigest(), 16)

def valor_estable(clave, lo: float, hi: float, salt: str) -> float:
    # Valor determinista y estable en [lo, hi] a partir de una clave (semilla 42).
    r = np.random.default_rng(_hash_u32(SEED, salt, clave))
    return float(r.uniform(lo, hi))

# Precio sintético estable por SKU (no aporta señal real; solo permite calcular 'ingreso').
skus_unicos = sorted(df["item_nbr"].unique().tolist())
precio_por_sku = {s: round(valor_estable(s, 1.5, 35.0, "precio"), 2) for s in skus_unicos}
lead_por_sku = {s: int(round(valor_estable(s, 2, 10, "lead"))) for s in skus_unicos}
print("Ejemplo precio SKU:", list(precio_por_sku.items())[:3])


Ejemplo precio SKU: [(96995, 9.69), (99197, 28.02), (103501, 18.27)]


## 6. Dominio VENTAS (mapeo real + factores marcados)

In [8]:
# === 6. VENTAS ===
v = pd.DataFrame()
v["fecha"] = df["date"].dt.strftime("%Y-%m-%d")
v["id_tienda"] = "T" + df["store_nbr"].astype(int).astype(str).str.zfill(2)
v["sku"] = "SKU-" + df["item_nbr"].astype(int).astype(str)
v["categoria"] = df["family"].astype(str)
v["unidades_vendidas"] = np.round(np.clip(df["unit_sales"].to_numpy(), 0, None).astype("float64"), 3)  # REAL (devoluciones->0)
v["precio_unitario"] = df["item_nbr"].map(precio_por_sku).astype("float64")                # SINTÉTICO
v["ingreso"] = (v["unidades_vendidas"] * v["precio_unitario"]).round(2)                     # DERIVADO (calculada)
v["en_promocion"] = df["onpromotion"].astype("int64")                                      # REAL
v["descuento_pct"] = 0.0                                                                    # SINTÉTICO (sin dato)
v["metodo_pago"] = "no_disponible"                                                         # SINTÉTICO
v["canal_venta"] = "tienda"                                                                # DERIVADO (Favorita = físico)
v["es_fin_de_semana"] = df["es_fin_de_semana"].astype("int64")                             # DERIVADO
v["dias_a_proximo_feriado"] = df["dias_a_proximo_feriado"].astype("int64")                 # DERIVADO
v = v[VENTAS.orden]
validar_conforme(v, "ventas")
print("VENTAS conforme:", v.shape)
print(v.head(3).to_string(index=False))


VENTAS conforme: (4713306, 13)
     fecha id_tienda        sku categoria  unidades_vendidas  precio_unitario  ingreso  en_promocion  descuento_pct   metodo_pago canal_venta  es_fin_de_semana  dias_a_proximo_feriado
2017-01-02       T01 SKU-103520 GROCERY I                1.0             2.10     2.10             0            0.0 no_disponible      tienda                 0                       0
2017-01-02       T01 SKU-105575 GROCERY I                3.0            25.93    77.79             0            0.0 no_disponible      tienda                 0                       0
2017-01-02       T01 SKU-105577 GROCERY I                1.0             6.53     6.53             0            0.0 no_disponible      tienda                 0                       0


## 7. Dominio ALMACÉN (target real + política de stock documentada)

In [9]:
# === 7. ALMACÉN ===
# Grano fecha×tienda×sku×día (igual que ventas). Objetivo demanda_dia = unit_sales (REAL).
a = pd.DataFrame()
a["fecha_dt"] = df["date"]
a["id_tienda"] = "T" + df["store_nbr"].astype(int).astype(str).str.zfill(2)
a["sku"] = "SKU-" + df["item_nbr"].astype(int).astype(str)
a["categoria"] = df["family"].astype(str)
a["item_nbr"] = df["item_nbr"].to_numpy()
a["demanda_dia"] = np.clip(np.round(df["unit_sales"].to_numpy()), 0, None).astype("int64")  # REAL
a = a.sort_values(["id_tienda", "sku", "fecha_dt"]).reset_index(drop=True)
grp = a.groupby(["id_tienda", "sku"], observed=True)

# demanda_diaria_promedio = media móvil trailing 28d por serie (shift(1) anti-fuga). REAL/DERIVADO.
def _trailing_mean(s):
    return s.shift(1).rolling(28, min_periods=1).mean()
a["demanda_diaria_promedio"] = (grp["demanda_dia"].transform(_trailing_mean)
                                 .fillna(a["demanda_dia"]).round(3))

# Política de inventario DETERMINISTA (documentada): lead por sku, min = dda_media*lead, max = min*factor.
lead = a["item_nbr"].map(lead_por_sku).astype("float64")
a["tiempo_reposicion_dias"] = lead.astype("int64")                                          # SINTÉTICO
dda = a["demanda_diaria_promedio"].clip(lower=0.1)
a["stock_minimo"] = (dda * lead).round(2)                                                    # DERIVADO
factor = a["item_nbr"].map(lambda s: valor_estable(s, 1.5, 2.5, "stockfactor")).astype("float64")
a["stock_maximo"] = (a["stock_minimo"] * factor).round(2)                                    # DERIVADO
# stock_actual: dientes de sierra por ciclo de reposición + ruido. El diente baja por DEBAJO del
# mínimo (para que existan quiebres reales -> 'riesgo_quiebre'/'reposicion_urgente' con dos clases)
# y el ruido puede superar el máximo (para que 'sobrestock' también tenga positivos). DERIVADO.
pos = grp.cumcount().to_numpy()
ciclo = np.maximum(lead.to_numpy(), 1.0)
frac = (pos % ciclo) / ciclo
rng_a = np.random.default_rng(_hash_u32(SEED, "almacen_stock"))
smin = a["stock_minimo"].to_numpy(); smax = a["stock_maximo"].to_numpy()
piso = 0.5 * smin                       # el diente cae hasta la mitad del mínimo
ruido = rng_a.normal(0.0, 0.06, len(a)) * np.maximum(smax, 1.0)
a["stock_actual"] = np.clip(smax - frac * (smax - piso) + ruido, 0.0, None).round(2)
stock_medio = ((a["stock_minimo"] + a["stock_maximo"]) / 2).clip(lower=0.1)
a["rotacion"] = (dda / stock_medio).round(4)                                                # DERIVADO
a["dias_de_cobertura"] = (a["stock_actual"] / dda).round(2)                                  # DERIVADO (calculada)
# zona_almacen A/B/C por terciles de rotación media del sku (ABC). DERIVADO.
rot_sku = a.groupby("sku", observed=True)["rotacion"].transform("mean")
a["zona_almacen"] = pd.qcut(rot_sku.rank(method="first"), 3, labels=["A", "B", "C"]).astype(str)
a["fecha"] = a["fecha_dt"].dt.strftime("%Y-%m-%d")
almacen = a[ALMACEN.orden].copy()
validar_conforme(almacen, "almacen")
print("ALMACÉN conforme:", almacen.shape)
print(almacen.head(3).to_string(index=False))


ALMACÉN conforme: (4713306, 13)
     fecha id_tienda         sku categoria  stock_actual  stock_minimo  stock_maximo  demanda_dia  demanda_diaria_promedio  dias_de_cobertura  rotacion  tiempo_reposicion_dias zona_almacen
2017-01-13       T01 SKU-1000866 GROCERY I         16.63           9.0         17.01            1                      1.0              16.63    0.0769                       9            A
2017-01-28       T01 SKU-1000866 GROCERY I         15.40           9.0         17.01            1                      1.0              15.40    0.0769                       9            A
2017-02-06       T01 SKU-1000866 GROCERY I         12.44           9.0         17.01            1                      1.0              12.44    0.0769                       9            A


## 8. Dominio COMPRAS (reposición anclada en demanda real + proxies marcados)

In [10]:
# === 8. COMPRAS ===
# Órdenes de reposición: demanda agregada por (sku, semana) entre todas las tiendas.
# id_proveedor derivado de la familia (supuesto documentado). cantidad_pedida = demanda del período (DERIVADO).
familias = sorted(df["family"].astype(str).unique().tolist())
prov_por_familia = {f: f"PROV-{i:02d}" for i, f in enumerate(familias, 1)}

base = pd.DataFrame({
    "fecha_dt": df["date"], "item_nbr": df["item_nbr"].to_numpy(),
    "family": df["family"].astype(str).to_numpy(),
    "unidades": np.clip(df["unit_sales"].to_numpy(), 0, None),
})
base["semana"] = base["fecha_dt"].dt.to_period("W-MON").dt.start_time
agg = (base.groupby(["semana", "item_nbr", "family"], observed=True)["unidades"]
            .sum().reset_index())

c = pd.DataFrame()
c["fecha_orden"] = agg["semana"].dt.strftime("%Y-%m-%d")                                     # DERIVADO
c["id_proveedor"] = agg["family"].map(prov_por_familia)                                      # SINTÉTICO (supuesto)
c["sku"] = "SKU-" + agg["item_nbr"].astype(int).astype(str)                                  # REAL
c["categoria"] = agg["family"]                                                              # REAL
c["cantidad_pedida"] = agg["unidades"].round(2)                                             # DERIVADO (target)
c["precio_unitario_compra"] = agg["item_nbr"].map(
    lambda s: round(valor_estable(s, 1.0, 25.0, "preciocompra"), 2))                        # SINTÉTICO
c["costo_total"] = (c["cantidad_pedida"] * c["precio_unitario_compra"]).round(2)            # DERIVADO (calculada)
# lead_time y cumplimiento con variación POR ORDEN (base por proveedor + ruido determinista):
# imprescindible para que las etiquetas 'entrega_con_retraso' y 'cumplimiento_alto' no sean de
# una sola clase (un valor constante por proveedor las degenera).
rng_c = np.random.default_rng(_hash_u32(SEED, "compras_orders"))
base_lead = agg["family"].map(lambda f: valor_estable(f, 4, 10, "leadprov")).to_numpy()
c["lead_time_dias"] = np.clip(np.round(base_lead + rng_c.integers(-3, 4, len(agg))), 1, None).astype("int64")  # SINTÉTICO
base_cumpl = agg["family"].map(lambda f: valor_estable(f, 0.88, 0.99, "cumpl")).to_numpy()
cumpl = np.clip(base_cumpl + rng_c.normal(0.0, 0.06, len(agg)), 0.55, 1.0)
# min() garantiza recibida <= pedida pese al redondeo (invariante de negocio: no llega más de lo pedido).
c["cantidad_recibida"] = np.minimum(c["cantidad_pedida"].to_numpy(),
                                    (c["cantidad_pedida"].to_numpy() * cumpl).round(2))       # SINTÉTICO
c["cumplimiento"] = (c["cantidad_recibida"] / c["cantidad_pedida"].clip(lower=0.1)).round(4)  # DERIVADO (calc)
c["metodo_pago"] = agg["family"].map(
    lambda f: ["contado", "credito_15", "credito_30"][_hash_u32(SEED, "pago", f) % 3])       # SINTÉTICO
c["descuento_volumen"] = agg["item_nbr"].map(
    lambda s: round(valor_estable(s, 0.0, 12.0, "descvol"), 2))                             # SINTÉTICO
compras = c[COMPRAS.orden].copy()
validar_conforme(compras, "compras")
print("COMPRAS conforme:", compras.shape)
print(compras.head(3).to_string(index=False))

# Libera memoria: df/train ya no se usan (v, almacen, compras están construidos).
import gc
del df, train, base, agg, a, c
gc.collect()


COMPRAS conforme: (128209, 12)
fecha_orden id_proveedor        sku categoria  cantidad_pedida  precio_unitario_compra  costo_total  lead_time_dias  cantidad_recibida  cumplimiento metodo_pago  descuento_volumen
 2016-12-27      PROV-13  SKU-99197 GROCERY I              6.0                   11.31        67.86               8                6.0        1.0000  credito_15               5.41
 2016-12-27      PROV-08 SKU-103501  CLEANING             60.0                    9.62       577.20               6               60.0        1.0000  credito_15               8.69
 2016-12-27      PROV-13 SKU-103520 GROCERY I             24.0                   24.61       590.64               7               19.4        0.8083  credito_15               4.19


0

## 9. Cortes temporales + etiqueta binaria (P75 train-only, anti-fuga)

In [11]:
# === 9. Cortes temporales y etiqueta demanda_alta (VENTAS) ===
from spc.models.automl import cortes_adaptativos

v_fechas = pd.to_datetime(v["fecha"])
cortes = cortes_adaptativos(v_fechas)
print("Cortes:", cortes.as_dict())

train_mask = v_fechas <= cortes.train_fin
valid_mask = (v_fechas >= cortes.valid_ini) & (v_fechas <= cortes.valid_fin)
test_mask = (v_fechas >= cortes.test_ini) & (v_fechas <= cortes.test_fin)

# P75 por categoría fijado SOLO en TRAIN, aplicado a los tres splits (anti-fuga).
p75_cat = v.loc[train_mask].groupby("categoria")["unidades_vendidas"].quantile(0.75)
umbral = v["categoria"].map(p75_cat)
umbral = umbral.fillna(v.loc[train_mask, "unidades_vendidas"].quantile(0.75))
v_label = (v["unidades_vendidas"] > umbral).astype("int64")

for nom, m in [("train", train_mask), ("valid", valid_mask), ("test", test_mask)]:
    y = v_label[m]
    print(f"{nom:6s}: n={len(y):>8,}  prevalencia demanda_alta={y.mean():.3f}")


Cortes: {'train': '<= 2017-07-14', 'valid': '2017-07-15 .. 2017-07-30', 'test': '2017-07-31 .. 2017-08-15'}


train : n=4,056,183  prevalencia demanda_alta=0.232
valid : n= 328,423  prevalencia demanda_alta=0.222
test  : n= 328,700  prevalencia demanda_alta=0.223


## 10. Balanceo de la clasificación con SMOTE — **solo train**, tras el corte temporal

SMOTE (SMOTENC) se aplica **únicamente** a la etiqueta binaria derivada, sobre el conjunto de
**entrenamiento** ya separado por fecha. Se comparan tres estrategias reutilizando
`spc.models.desbalance`; SMOTE se adopta solo si supera a la costo-sensible en PR-AUC (VALID).

In [12]:
# === 10. SMOTE (SMOTENC) solo en TRAIN ===
from sklearn.metrics import average_precision_score
from spc.models.desbalance import construir_estrategia, _elegir_estrategia, seleccionar_umbral, ESTRATEGIAS

# Matriz de features para la clasificación (categóricas como dtype category -> SMOTENC='auto').
feat_cat = ["categoria", "id_tienda"]
feat_num = ["precio_unitario", "en_promocion", "descuento_pct", "es_fin_de_semana", "dias_a_proximo_feriado"]
X_all = v[feat_cat + feat_num].copy()
X_all["_month"] = pd.to_datetime(v["fecha"]).dt.month
X_all["_dow"] = pd.to_datetime(v["fecha"]).dt.dayofweek
for cc in feat_cat:
    X_all[cc] = X_all[cc].astype("category")
y_all = v_label

# Submuestra el TRAIN para la comparación de estrategias (SMOTENC+LGBM ×3 es costoso a escala
# de millones de filas). Es una decisión de DEMO: los CSV exportados quedan a escala completa;
# aquí solo se compara la estrategia de desbalance sobre un train representativo. VALID completo.
CAP_CLASIF_TRAIN = 200_000
Xtr_full, ytr_full = X_all[train_mask], y_all[train_mask]
if len(Xtr_full) > CAP_CLASIF_TRAIN:
    _idx = Xtr_full.sample(CAP_CLASIF_TRAIN, random_state=SEED).index
    Xtr, ytr = Xtr_full.loc[_idx], ytr_full.loc[_idx]
    print(f"(train submuestreado a {CAP_CLASIF_TRAIN:,} filas para la demo de clasificación)")
else:
    Xtr, ytr = Xtr_full, ytr_full
Xva, yva = X_all[valid_mask], y_all[valid_mask]

n_pos = int(ytr.sum()); n_neg = int((ytr == 0).sum())
spw = n_neg / max(1, n_pos)
print(f"TRAIN: pos={n_pos:,} neg={n_neg:,} scale_pos_weight={spw:.2f}")

metricas_valid = {}
modelos = {}
for est in ESTRATEGIAS:
    if est == "smote" and n_pos < 6:
        print("smote: omitido (n_pos<6, insuficiente para k_neighbors)"); continue
    modelo = construir_estrategia(est, SEED, usar_gpu=False, scale_pos_weight=spw)
    modelo.fit(Xtr, ytr)
    prob_va = modelo.predict_proba(Xva)[:, 1]
    prauc = average_precision_score(yva, prob_va)
    metricas_valid[est] = {"PR_AUC": float(prauc), "Recall": float(((prob_va >= 0.5) & (yva == 1)).sum() / max(1, (yva == 1).sum()))}
    modelos[est] = modelo
    print(f"{est:14s}: PR-AUC(valid)={prauc:.4f}")

elegida, criterio = _elegir_estrategia(metricas_valid)
print("\nEstrategia elegida (más simple dentro de tolerancia):", elegida)
print("Regla:", criterio["regla"])


(train submuestreado a 200,000 filas para la demo de clasificación)
TRAIN: pos=46,337 neg=153,663 scale_pos_weight=3.32


sin_remuestreo: PR-AUC(valid)=0.5370


costo_sensible: PR-AUC(valid)=0.5263


smote         : PR-AUC(valid)=0.4492

Estrategia elegida (más simple dentro de tolerancia): sin_remuestreo
Regla: estrategia mas simple (sin_remuestreo < costo_sensible < smote) cuya PR-AUC en VALID esta dentro de 0.005 de la mejor; SMOTE solo se adopta si SUPERA a la costo-sensible por mas de esa tolerancia


## 11. Distribución antes / después del balanceo (solo train)

In [13]:
# === 11. Antes vs después de SMOTE (solo train) ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

antes = ytr.value_counts().sort_index()
print("TRAIN antes  :", dict(antes), "| prevalencia", round(ytr.mean(), 3))

despues = None
if "smote" in modelos:
    from imblearn.over_sampling import SMOTENC
    cat_idx = [X_all.columns.get_loc(c) for c in feat_cat]
    sm = SMOTENC(categorical_features=cat_idx, random_state=SEED)
    Xrs, yrs = sm.fit_resample(Xtr, ytr)
    despues = pd.Series(yrs).value_counts().sort_index()
    print("TRAIN después:", dict(despues), "| prevalencia", round(float(pd.Series(yrs).mean()), 3))
    print("VALID/TEST: intactos (nunca se remuestrean).")

fig, ax = plt.subplots(1, 2 if despues is not None else 1, figsize=(9, 3.2))
ax = np.atleast_1d(ax)
ax[0].bar([0, 1], antes.reindex([0, 1], fill_value=0).values, color=["#4C78A8", "#F58518"])
ax[0].set_title("TRAIN antes de SMOTE"); ax[0].set_xticks([0, 1])
if despues is not None:
    ax[1].bar([0, 1], despues.reindex([0, 1], fill_value=0).values, color=["#4C78A8", "#F58518"])
    ax[1].set_title("TRAIN después de SMOTENC"); ax[1].set_xticks([0, 1])
plt.tight_layout(); plt.show()


TRAIN antes  : {0: np.int64(153663), 1: np.int64(46337)} | prevalencia 0.232


TRAIN después: {0: np.int64(153663), 1: np.int64(153663)} | prevalencia 0.5
VALID/TEST: intactos (nunca se remuestrean).


C:\Users\val_f\AppData\Local\Temp\ipykernel_35428\4182520719.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 12. Entrenamiento y evaluación (regresión WAPE + clasificación PR-AUC vs baseline)

In [14]:
# === 12. Modelos de referencia (demo; el motor del SPC hace la versión completa) ===
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score

def wape(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    d = np.abs(y_true).sum()
    return float(np.abs(y_true - y_pred).sum() / d) if d > 0 else float("nan")

# --- Regresión de demanda (log1p) con muestreo para velocidad de la demo ---
# Nivel por serie: codificación media-en-TRAIN de (tienda,sku) — ajustada SOLO en train (anti-fuga).
# Sin señal de nivel por serie un modelo tabular no puede superar al promedio por serie (baseline).
serie = v["id_tienda"].astype(str) + "|" + v["sku"].astype(str)
media_serie_tr = v.loc[train_mask].assign(_s=serie[train_mask]).groupby("_s")["unidades_vendidas"].mean()
media_global_tr = float(v.loc[train_mask, "unidades_vendidas"].mean())
serie_media = serie.map(media_serie_tr).fillna(media_global_tr).to_numpy()

num_reg = ["precio_unitario", "en_promocion", "descuento_pct", "es_fin_de_semana", "dias_a_proximo_feriado"]
Xr = v[num_reg].copy()
Xr["cat"] = v["categoria"].astype("category").cat.codes
Xr["tienda"] = v["id_tienda"].astype("category").cat.codes
Xr["month"] = pd.to_datetime(v["fecha"]).dt.month
Xr["serie_media"] = serie_media          # nivel por serie (train-only)
yr = np.log1p(v["unidades_vendidas"].to_numpy())

tr_idx = np.where(train_mask.to_numpy())[0]
if len(tr_idx) > 300_000:
    tr_idx = np.random.default_rng(SEED).choice(tr_idx, 300_000, replace=False)
te_idx = np.where(test_mask.to_numpy())[0]

reg = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.06, max_leaf_nodes=63, random_state=SEED)
reg.fit(Xr.iloc[tr_idx], yr[tr_idx])
pred_te = np.expm1(reg.predict(Xr.iloc[te_idx]))
y_te = v["unidades_vendidas"].to_numpy()[te_idx]

# Baseline naïve: media de demanda de la serie (tienda,sku) en TRAIN.
base_te = serie.iloc[te_idx].map(media_serie_tr).fillna(media_global_tr).to_numpy()

print(f"WAPE modelo (TEST):  {wape(y_te, pred_te):.4f}")
print(f"WAPE baseline naïve: {wape(y_te, base_te):.4f}")

# --- Clasificación: PR-AUC del modelo elegido en TEST ---
Xte_cls, yte_cls = X_all[test_mask], y_all[test_mask]
prob_te = modelos[elegida].predict_proba(Xte_cls)[:, 1]
print(f"\nClasificación '{elegida}': PR-AUC(TEST)={average_precision_score(yte_cls, prob_te):.4f} "
      f"| prevalencia TEST={yte_cls.mean():.3f} (azar)")


WAPE modelo (TEST):  0.4784
WAPE baseline naïve: 0.5107



Clasificación 'sin_remuestreo': PR-AUC(TEST)=0.5023 | prevalencia TEST=0.223 (azar)


## 13. Validación de coherencia de los datos sintéticos/derivados

In [15]:
# === 13. Reglas de coherencia ===
chks = {}
# Post-SMOTE (si aplica): categóricas válidas + numéricas en rango del train real.
if "smote" in modelos:
    chks["smote_categorias_validas"] = bool(set(pd.Series(Xrs["categoria"]).unique()) <= set(Xtr["categoria"].cat.categories))
    chks["smote_precio_no_negativo"] = bool((Xrs["precio_unitario"] >= 0).all())
    chks["smote_valid_test_intactos"] = True  # nunca se tocan por construcción
# Dominios: rangos de negocio.
chks["ventas_unidades_no_neg"] = bool((v["unidades_vendidas"] >= 0).all())
chks["ventas_categorias_conocidas"] = bool(set(v["categoria"].unique()) <= set(items["family"].astype(str)))
chks["almacen_stock_min_le_max"] = bool((almacen["stock_minimo"] <= almacen["stock_maximo"]).all())
chks["almacen_demanda_entera"] = bool(np.array_equal(almacen["demanda_dia"], np.round(almacen["demanda_dia"])))
chks["compras_cumplimiento_0_1"] = bool(((compras["cumplimiento"] >= 0) & (compras["cumplimiento"] <= 1.001)).all())
chks["compras_recibida_le_pedida"] = bool((compras["cantidad_recibida"] <= compras["cantidad_pedida"] + 1e-6).all())
for k, ok in chks.items():
    print(("OK  " if ok else "FALLA") + f"  {k}")
assert all(chks.values()), "Alguna regla de coherencia falló"


OK    smote_categorias_validas
OK    smote_precio_no_negativo
OK    smote_valid_test_intactos
OK    ventas_unidades_no_neg
OK    ventas_categorias_conocidas
OK    almacen_stock_min_le_max
OK    almacen_demanda_entera
OK    compras_cumplimiento_0_1
OK    compras_recibida_le_pedida


## 14. Por qué SMOTE **no** aplica a la regresión (evidencia)

SMOTE/SMOTENC exigen una **etiqueta discreta**: interpolan entre vecinos de la **clase minoritaria**.
Con un target **continuo** (`unidades_vendidas`) no hay clases, así que la librería falla o exige
binarizar el objetivo — y binarizar destruye la variable que se quiere predecir.

In [16]:
# === 14. Demostración empírica ===
from imblearn.over_sampling import SMOTE

# Muestra de train para la demostración (rápida).
Xdemo = Xr.iloc[tr_idx].to_numpy(dtype=float)
y_cont = v["unidades_vendidas"].to_numpy()[tr_idx]

# 14a. SMOTE sobre target continuo -> error (no hay clases que interpolar).
try:
    SMOTE(random_state=SEED).fit_resample(Xdemo, y_cont)
    print("Inesperado: no debería remuestrear un objetivo continuo.")
except Exception as e:
    print("SMOTE sobre target continuo FALLA (correcto):", type(e).__name__)
    print("  ->", str(e).splitlines()[0][:140])

# 14b. Forzar SMOTE binarizando el target infla artificialmente la cola alta de demanda.
from scipy.stats import ks_2samp
umb = np.quantile(y_cont, 0.75)
y_bin = (y_cont > umb).astype(int)
_, y_bin_rs = SMOTE(random_state=SEED).fit_resample(Xdemo, y_bin)
# La demanda "reconstruida" tras oversamplear la clase alta: duplica valores altos -> distorsión.
demanda_original = y_cont
demanda_binmote = np.r_[y_cont, y_cont[y_bin == 1][: (y_bin_rs == 1).sum() - int(y_bin.sum())]]
ks = ks_2samp(demanda_original, demanda_binmote)
print(f"\nKS demanda (original vs binarizar+SMOTE): stat={ks.statistic:.3f} (mayor = más distorsión de la distribución real)")
print("Conclusión: el conteo con exceso de ceros se modela con objetivos Tweedie/Poisson + log1p")
print("(ya presentes en spc.models.nucleo), NO con SMOTE.")


SMOTE sobre target continuo FALLA (correcto): ValueError
  -> Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continu



KS demanda (original vs binarizar+SMOTE): stat=0.143 (mayor = más distorsión de la distribución real)
Conclusión: el conteo con exceso de ceros se modela con objetivos Tweedie/Poisson + log1p
(ya presentes en spc.models.nucleo), NO con SMOTE.


## 15. Exportación de los datasets transformados + MANIFIESTO

In [17]:
# === 15. Exportar CSV + MANIFIESTO ===
p_v = OUT_DATOS / "ventas_favorita.csv"
p_a = OUT_DATOS / "almacen_favorita.csv"
p_c = OUT_DATOS / "compras_favorita.csv"
v.to_csv(p_v, index=False)
almacen.to_csv(p_a, index=False)
compras.to_csv(p_c, index=False)

# Manifiesto de procedencia por campo (REAL/DERIVADO/SINTÉTICO).
proc = {
    "ventas": {"fecha": "REAL", "id_tienda": "REAL", "sku": "REAL", "categoria": "REAL",
               "unidades_vendidas": "REAL", "precio_unitario": "SINTETICO", "ingreso": "DERIVADO",
               "en_promocion": "REAL", "descuento_pct": "SINTETICO", "metodo_pago": "SINTETICO",
               "canal_venta": "DERIVADO", "es_fin_de_semana": "DERIVADO", "dias_a_proximo_feriado": "DERIVADO"},
    "almacen": {"fecha": "REAL", "id_tienda": "REAL", "sku": "REAL", "categoria": "REAL",
                "stock_actual": "DERIVADO", "stock_minimo": "DERIVADO", "stock_maximo": "DERIVADO",
                "demanda_dia": "REAL", "demanda_diaria_promedio": "DERIVADO", "dias_de_cobertura": "DERIVADO",
                "rotacion": "DERIVADO", "tiempo_reposicion_dias": "SINTETICO", "zona_almacen": "DERIVADO"},
    "compras": {"fecha_orden": "DERIVADO", "id_proveedor": "SINTETICO", "sku": "REAL", "categoria": "REAL",
                "cantidad_pedida": "DERIVADO", "precio_unitario_compra": "SINTETICO", "costo_total": "DERIVADO",
                "lead_time_dias": "SINTETICO", "cantidad_recibida": "SINTETICO", "cumplimiento": "DERIVADO",
                "metodo_pago": "SINTETICO", "descuento_volumen": "SINTETICO"},
}
filas_manif = []
tamanos = {"ventas": len(v), "almacen": len(almacen), "compras": len(compras)}
for dom, campos in proc.items():
    for campo, marca in campos.items():
        filas_manif.append({"dominio": dom, "campo": campo, "procedencia": marca,
                            "filas": tamanos[dom], "semilla": SEED})
manif = pd.DataFrame(filas_manif)
manif.to_csv(OUT_DATOS / "MANIFIESTO.csv", index=False)

# Metadatos de la corrida.
(OUT_DATOS / "metadatos.json").write_text(json.dumps({
    "fuente": "data_smote (Kaggle Corporación Favorita)",
    "submuestra": resumen_submuestra,
    "cortes": cortes.as_dict(),
    "estrategia_desbalance_elegida": elegida,
    "archivos": [p.name for p in (p_v, p_a, p_c)],
}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("Exportado a", OUT_DATOS)
for p in (p_v, p_a, p_c):
    print(f"  {p.name}: {p.stat().st_size/1e6:.1f} MB")
print("Procedencia (conteo):")
print(manif.groupby("procedencia")["campo"].count().to_string())


Exportado a D:\UPAO\IX\Taller Integrador I\sistema_prediccion_comercializacion\data\favorita_real
  ventas_favorita.csv: 401.7 MB
  almacen_favorita.csv: 375.4 MB
  compras_favorita.csv: 11.7 MB
Procedencia (conteo):
procedencia
DERIVADO     15
REAL         13
SINTETICO    10


## 16. Archivos de prueba compatibles con Excel y JSON (API `/v2` y `/v3`)

In [18]:
# === 16. Plantillas Excel + payloads JSON ===
from spc.api.ingest.dominios_excel import generar_excel, leer_excel

dfs = {"ventas": v, "almacen": almacen, "compras": compras}
N_MUESTRA = 300  # filas de ejemplo por dominio para los archivos de prueba

for dom, d in dfs.items():
    muestra = d.head(N_MUESTRA).to_dict(orient="records")
    # Excel (hoja 'datos' + 'instrucciones'); round-trip de verificación.
    xlsx = generar_excel(dom, muestra)
    (OUT_EJEMPLOS / f"{dom}.xlsx").write_bytes(xlsx)
    releido = leer_excel(xlsx, dom)
    assert len(releido) == len(muestra), f"round-trip Excel {dom} falló"
    # JSON /v2 (con horizon) y /v3 (solo rows).
    (OUT_EJEMPLOS / f"{dom}_v2.json").write_text(
        json.dumps({"rows": muestra, "horizon": 14}, ensure_ascii=False, default=str), encoding="utf-8")
    (OUT_EJEMPLOS / f"{dom}_v3.json").write_text(
        json.dumps({"rows": muestra}, ensure_ascii=False, default=str), encoding="utf-8")
    print(f"{dom:8s}: xlsx round-trip OK ({len(releido)} filas), JSON v2/v3 escritos")

print("\nEjemplos en", OUT_EJEMPLOS)
print("Payload v2 (1 fila ventas):")
print(json.dumps({"rows": v.head(1).to_dict(orient="records"), "horizon": 14}, ensure_ascii=False, default=str, indent=2)[:600])


ventas  : xlsx round-trip OK (300 filas), JSON v2/v3 escritos


almacen : xlsx round-trip OK (300 filas), JSON v2/v3 escritos
compras : xlsx round-trip OK (300 filas), JSON v2/v3 escritos

Ejemplos en D:\UPAO\IX\Taller Integrador I\sistema_prediccion_comercializacion\examples\favorita_real
Payload v2 (1 fila ventas):
{
  "rows": [
    {
      "fecha": "2017-01-02",
      "id_tienda": "T01",
      "sku": "SKU-103520",
      "categoria": "GROCERY I",
      "unidades_vendidas": 1.0,
      "precio_unitario": 2.1,
      "ingreso": 2.1,
      "en_promocion": 0,
      "descuento_pct": 0.0,
      "metodo_pago": "no_disponible",
      "canal_venta": "tienda",
      "es_fin_de_semana": 0,
      "dias_a_proximo_feriado": 0
    }
  ],
  "horizon": 14
}


## 17. Conclusiones y checklist de compatibilidad

**Generado**
- `data/favorita_real/{ventas,almacen,compras}_favorita.csv` — conformes a `esquemas.py` (`validar_conforme` OK).
- `data/favorita_real/MANIFIESTO.csv` — procedencia REAL / DERIVADO / SINTÉTICO por campo.
- `examples/favorita_real/{dominio}.xlsx` + `{dominio}_v2.json` + `{dominio}_v3.json`.

**Checklist**
1. ✅ Conformidad de esquema (columnas exactas y en orden) para los 3 dominios.
2. ✅ Round-trip Excel (`generar_excel` → `leer_excel`) sin pérdida.
3. ✅ Corte **temporal** (no aleatorio); P75 y SMOTE ajustados **solo en TRAIN**.
4. ✅ SMOTE (SMOTENC) **solo** en la etiqueta binaria derivada, **solo train**; valid/test intactos.
5. ✅ SMOTE **no** aplicado al target continuo (demostrado empíricamente §14).
6. ✅ Métrica honesta: WAPE del modelo comparado contra baseline naïve.
7. ✅ Coherencia de sintéticos verificada (§13).

**Prueba end-to-end sugerida** (fuera del notebook): levantar la API (`uvicorn spc.api.main:app`) y
enviar los JSON de `examples/favorita_real/` a `POST /v2/{ventas,compras,almacen}` y
`POST /v3/{sales,purchases,inventory}` → esperar **200** con los bloques/reportes.

**Nota de honestidad.** Solo **VENTAS** proviene casi íntegramente de datos reales. En **ALMACÉN** el
target (`demanda_dia`) es real, pero el estado de stock es una **política simulada**. **COMPRAS** es
**mayormente sintético**: solo `cantidad_pedida` está anclada en la demanda real agregada. Todo está
marcado en el `MANIFIESTO.csv`.
